# LOKLOK LLM extension — gated Colab execution

This notebook never bypasses the project’s approval gate. It runs safe tests by default and exports an integration-ready results candidate only after a researcher has explicitly opened every gate in a separate, sanitized working copy.


In [ ]:
# Upload a sanitized ZIP of a *working copy* containing only 02_code/llm_agent_study.
from google.colab import files
from pathlib import Path
import zipfile, json, os, shutil
uploaded=files.upload(); archive=next(iter(uploaded))
ROOT=Path('/content/loklok_work'); ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z: z.extractall(ROOT)
study=next(ROOT.rglob('llm_agent_study'))
os.chdir(study)
protocol=json.loads((study/'llm_extension_protocol.json').read_text())
print('Protocol status:',protocol['status'])
print('External model upload authorised:',protocol['external_model_upload_authorized'])
print('Execution gate:',protocol.get('execution_gate',{}))


In [ ]:
# Safe verification only. Fixture/stub output is never thesis evidence.
!pip -q install pytest pandas numpy scipy matplotlib
!pytest -q experiment_work/tests
!python -m experiment_work.run_networks --help
!python -m experiment_work.export_task_responses --help
!python -m experiment_work.package_run --help
!python -m experiment_work.render_results --help
!python -m experiment_work.hallucination_diagnostics --help
# EFI/PSD/PDS estimators must pass their regression suite and the detector self-check
# inside the pytest run above before any live execution may be approved.


In [ ]:
# Live execution is deliberately blocked until independent approvals are recorded in the uploaded working copy.
LIVE_EXECUTION_APPROVED=False  # Change only after all listed approvals are present and reviewed.
required=['external_model_upload_authorized','model_version','backend_commit','run_count','minimum_precision_target']
missing=[k for k in required if not protocol.get(k)]
if not LIVE_EXECUTION_APPROVED or protocol.get('status')!='approved_for_execution' or missing:
    raise RuntimeError('LIVE RUN BLOCKED. Missing or unapproved: '+', '.join(missing)+'. Fixture output must not be used as thesis results.')
# Once approved, use the project’s frozen plan and independently approved key pack. Keep credentials in Colab Secrets.
# !python -m experiment_work.run_networks --plan FROZEN_PLAN.json --keys-dir APPROVED_KEYS --out-dir run_live --base-url YOUR_ISOLATED_BACKEND --provider PROVIDER --model MODEL
# !python -m experiment_work.export_task_responses --run-dir run_live --task-bank experiment_work/task_bank.csv --key-dir APPROVED_KEYS --out run_live/responses.json
# !python -m experiment_work.comparative_analysis --help
# !python -m experiment_work.package_run --run-dir run_live --responses run_live/responses.json --plan run_live/plan.json --analysis run_live/analysis.json
## Hallucination & divergence diagnostics (Section 5.6.11). Supply real anchor text per
## agent only when the qualitative anchor is approved; see hallucination.txt. The multilingual
## embeddings for PDS are produced here, not computed locally, so the model hook below is
## only installed/used in this gated live cell after approval.
# !pip -q install sentence-transformers
# import json, time
# from sentence_transformers import SentenceTransformer
# from pathlib import Path
# embedder = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
# anchors = { <agent>: <approved anchor text> }  # researcher-provided only
# anchors_json = 'run_live/approved_anchors.json'
# Path(anchors_json).write_text(json.dumps(anchors, ensure_ascii=False), encoding='utf-8')
# !python -m experiment_work.hallucination_diagnostics --run-dir run_live \
#   --register experiment_work/evidence_register.json \
#   --content experiment_work/evidence_content_seed.json \
#   --anchors run_live/approved_anchors.json \
#   --out run_live/hallucination_diagnostics.json
# !python -m experiment_work.render_results --run-dir run_live --plan run_live/plan.json --analysis run_live/analysis.json --responses run_live/responses.json --out thesis_results_candidate


In [ ]:
# Download only an approved, hashed results candidate; import it into a separate local working copy for review.
# shutil.make_archive('/content/LLM_RESULTS_CANDIDATE','zip','thesis_results_candidate')
# files.download('/content/LLM_RESULTS_CANDIDATE.zip')
print('No result bundle is created unless the live execution cell completes successfully.')
